# 04 · 설명가능성 (SHAP)

최적 모델의 상위 센서 → "어느 공정 단계가 불량을 주도하는가" 인사이트.

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from src.data import load_secom
from src.features import build_preprocessor
X, y, ts = load_secom()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

## 전처리 (train에서만 fit) + 살아남은 원본 센서명 복원

In [ ]:
pre = build_preprocessor().fit(Xtr)
Xtr_p, Xte_p = pre.transform(Xtr), pre.transform(Xte)
keep_missing = pre.named_steps["drop_missing"].keep_
cols1 = X.columns[keep_missing]
feat_names = cols1[pre.named_steps["var"].get_support()]
print("features after preprocessing:", len(feat_names))

## 모델 학습 (RandomForest)

In [ ]:
clf = RandomForestClassifier(n_estimators=400, class_weight="balanced_subsample",
                             n_jobs=-1, random_state=42).fit(Xtr_p, ytr)

## SHAP summary

In [ ]:
import shap
expl = shap.TreeExplainer(clf)
sv = expl.shap_values(Xte_p)
sv_pos = sv[1] if isinstance(sv, list) else sv
shap.summary_plot(sv_pos, Xte_p, feature_names=list(feat_names), max_display=20)

## 상위 센서 → 공정 해석

센서 ID를 실제 공정 단계(장비/스텝)에 매핑하면 공정 인사이트가 된다.

In [ ]:
imp = pd.Series(np.abs(sv_pos).mean(0), index=feat_names).sort_values(ascending=False)
print(imp.head(15))
# TODO: sensor id -> 공정 단계(증착/식각/CMP/포토) 매핑 = 도메인 지식 결합 지점